# Example 12.5 / 4: Feasibility Cuts and Benders Variants

Companion notebook for **12.5 Decomposition** in the AMPL lab Documentation, section "Feasibility Cuts and Benders Variants". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Feasibility Cuts and Benders Variants

Scenario cuts must use probabilities, signs,
and first-stage coefficients consistent with the extensive form. Classical linear Benders relies
on LP duality; integer or nonconvex subproblems need a separately proved logic-based or
problem-specific cut.

A single aggregated cut keeps the master small. Multi-cut Benders adds one cut per scenario and
can tighten it faster at the cost of more rows. Track per-scenario recourse so indexing errors do
not cancel in an aggregate. An infeasible subproblem needs a valid feasibility cut, often from an
extreme ray; a penalty slack changes the model unless exactness is established.

Treat an unbounded or failed subproblem as a separate state, never as zero recourse.

When outsourcing is not allowed, the subproblem at capacity 0 and demand 8 is infeasible. With
AMPL presolve off, HiGHS returns a dual ray in the `.dunbdd` suffix, as in the AMPL Development
Tutorial. Following the tutorial's convention, the ray gives the feasibility cut
`Supply.dunbdd * demand + CapacityUse.dunbdd * x <= 0`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand >= 0;                                        # units demanded
param capacity_fixed >= 0;                                # capacity fixed by the master
param max_outsource >= 0;                                 # outsourcing limit; 0 = not allowed
suffix dunbdd;                                            # dual ray of an infeasible LP
var Produce >= 0;                                         # units made in-house
var Idle >= 0;                                            # unused capacity
var Outsource >= 0, <= max_outsource;                     # units bought outside
minimize RecourseCost: 3 * Outsource;                     # outsourcing costs 3 per unit
subject to Supply: Produce + Outsource >= demand;         # meet demand
subject to CapacityUse: Produce + Idle = capacity_fixed;  # capacity split into production and idle

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 8
ampl.param["capacity_fixed"] = 0
ampl.param["max_outsource"] = 0

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="highs", verbose=False)
ray_supply = ampl.con["Supply"].get_values("dunbdd").to_list()[0]
ray_capacity = ampl.con["CapacityUse"].get_values("dunbdd").to_list()[0]
print(ampl.solve_result, "Supply.dunbdd", ray_supply, "CapacityUse.dunbdd", ray_capacity)
print(f"feasibility cut: {ray_supply} * 8 + {ray_capacity} * x <= 0")
ampl.param["capacity_fixed"] = 8
ampl.solve(solver="highs", verbose=False)
print("capacity 8:", ampl.solve_result)

infeasible Supply.dunbdd 1 CapacityUse.dunbdd -1
feasibility cut: 1 * 8 + -1 * x <= 0
capacity 8: solved


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param demand := 8;
param capacity_fixed := 0;
param max_outsource := 0;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0, presolve 0;
solve;
display solve_result, Supply.dunbdd, CapacityUse.dunbdd;
let capacity_fixed := 8;
solve;
display solve_result;

solve_result = infeasible
Supply.dunbdd = 1
CapacityUse.dunbdd = -1

solve_result = solved

